In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\dataa\Downloads\HW4_Retail_Orders.csv")
print(df.head())
print(df.shape)
print(df.columns)
print(df.dtypes)


  OrderID         Product         Category Quantity  UnitPrice Region
0   O1001  Wireless Mouse      Electronics        2      24.95  South
1   O1002        Notebook  Office Supplies        5       4.50   East
2   O1003       USB Drive      Electronics        3      15.00  North
3   O1004       Desk Lamp             Home        1      39.99   West
4   O1005    Water Bottle             Home        4      12.50  South
(31, 6)
Index(['OrderID', 'Product', 'Category', 'Quantity', 'UnitPrice', 'Region'], dtype='str')
OrderID          str
Product          str
Category         str
Quantity         str
UnitPrice    float64
Region           str
dtype: object


In [3]:
df.info()
print(df.describe())
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   OrderID    31 non-null     str    
 1   Product    31 non-null     str    
 2   Category   30 non-null     str    
 3   Quantity   31 non-null     str    
 4   UnitPrice  30 non-null     float64
 5   Region     30 non-null     str    
dtypes: float64(1), str(5)
memory usage: 1.6 KB
        UnitPrice
count   30.000000
mean    38.934667
std     54.388390
min      0.000000
25%      8.990000
50%     16.875000
75%     43.747500
max    189.990000
OrderID      0
Product      0
Category     1
Quantity     0
UnitPrice    1
Region       1
dtype: int64
Duplicates: 1


Quantity has a word in it, so it is not all numbers.Some cells are empty.There is a repeated row in the data.

In [4]:
clean_df = df.copy()
before = len(clean_df)
clean_df = clean_df.drop_duplicates()
print("Duplicates removed:", before - len(clean_df))

Duplicates removed: 1


In [5]:
clean_df["Category"] = clean_df["Category"].fillna("Unknown")
clean_df["Region"] = clean_df["Region"].fillna("Unknown")
print(clean_df[["Category", "Region"]].isna().sum())

Category    0
Region      0
dtype: int64


In [6]:
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["UnitPrice"] = pd.to_numeric(clean_df["UnitPrice"], errors="coerce")
print(clean_df[["Quantity", "UnitPrice"]].isna().sum())
before = len(clean_df)
clean_df = clean_df.dropna(subset=["Quantity", "UnitPrice"])
print("Missing numbers removed:", before - len(clean_df))

Quantity     1
UnitPrice    1
dtype: int64
Missing numbers removed: 2


In [7]:
before = len(clean_df)
clean_df = clean_df[(clean_df["Quantity"] > 0) & (clean_df["UnitPrice"] > 0)].copy()
print("Zero or negative values removed:", before - len(clean_df))
clean_df["Revenue"] = clean_df["Quantity"] * clean_df["UnitPrice"]
print(clean_df.shape)
print(clean_df.isna().sum())
print("Duplicates:", clean_df.duplicated().sum())
print(clean_df.head())


Zero or negative values removed: 2
(26, 7)
OrderID      0
Product      0
Category     0
Quantity     0
UnitPrice    0
Region       0
Revenue      0
dtype: int64
Duplicates: 0
  OrderID         Product         Category  Quantity  UnitPrice Region  \
0   O1001  Wireless Mouse      Electronics       2.0      24.95  South   
1   O1002        Notebook  Office Supplies       5.0       4.50   East   
2   O1003       USB Drive      Electronics       3.0      15.00  North   
3   O1004       Desk Lamp             Home       1.0      39.99   West   
4   O1005    Water Bottle             Home       4.0      12.50  South   

   Revenue  
0    49.90  
1    22.50  
2    45.00  
3    39.99  
4    50.00  


I changed the columns to numbers because some values were saved as text.
A quantity or price of zero or less should not count as a normal sale. It would affect the total.
There are 26 rows left. I removed 1 repeated row, 2 rows with missing or bad numbers, and 2 rows with   zero or negative values.

In [8]:
print(clean_df[["OrderID", "Product", "Quantity", "UnitPrice", "Revenue"]])
south_orders = clean_df[clean_df["Region"] == "South"]
print(south_orders)

   OrderID         Product  Quantity  UnitPrice  Revenue
0    O1001  Wireless Mouse       2.0      24.95    49.90
1    O1002        Notebook       5.0       4.50    22.50
2    O1003       USB Drive       3.0      15.00    45.00
3    O1004       Desk Lamp       1.0      39.99    39.99
4    O1005    Water Bottle       4.0      12.50    50.00
5    O1006        Keyboard       2.0      45.00    90.00
6    O1007            Pens      10.0       2.25    22.50
7    O1008         Monitor       1.0     189.99   189.99
8    O1009  Desk Organizer       2.0      18.75    37.50
9    O1010      Headphones       2.0      59.99   119.98
10   O1011        Notebook       6.0       4.50    27.00
11   O1012    Water Bottle       3.0      12.50    37.50
13   O1014       Mouse Pad       4.0       8.99    35.96
14   O1015       Desk Lamp       2.0      39.99    79.98
15   O1016            Pens       8.0       2.25    18.00
19   O1020      Headphones       1.0      59.99    59.99
20   O1021    Water Bottle     

In [12]:
electronics_orders = clean_df[(clean_df["Category"] == "Electronics") & (clean_df["Revenue"] > 100)]
print(electronics_orders)
print(clean_df.loc[:, ["OrderID", "Product", "Revenue"]])
print(clean_df.iloc[:3, :3])

   OrderID     Product     Category  Quantity  UnitPrice Region  Revenue
7    O1008     Monitor  Electronics       1.0     189.99  North   189.99
9    O1010  Headphones  Electronics       2.0      59.99  South   119.98
27   O1028     Monitor  Electronics       1.0     189.99   West   189.99
29   O1030  Headphones  Electronics       2.0      59.99   West   119.98
   OrderID         Product  Revenue
0    O1001  Wireless Mouse    49.90
1    O1002        Notebook    22.50
2    O1003       USB Drive    45.00
3    O1004       Desk Lamp    39.99
4    O1005    Water Bottle    50.00
5    O1006        Keyboard    90.00
6    O1007            Pens    22.50
7    O1008         Monitor   189.99
8    O1009  Desk Organizer    37.50
9    O1010      Headphones   119.98
10   O1011        Notebook    27.00
11   O1012    Water Bottle    37.50
13   O1014       Mouse Pad    35.96
14   O1015       Desk Lamp    79.98
15   O1016            Pens    18.00
19   O1020      Headphones    59.99
20   O1021    Water Bot

In [11]:
print("Total revenue:", round(clean_df["Revenue"].sum(), 2))
print("Average revenue:", round(clean_df["Revenue"].mean(), 2))
print("Highest revenue:", round(clean_df["Revenue"].max(), 2))
print(clean_df["Category"].value_counts())
print(clean_df[["Quantity", "UnitPrice", "Revenue"]].describe())
print("Orders over $100:", (clean_df["Revenue"] > 100).sum())

Total revenue: 1576.55
Average revenue: 60.64
Highest revenue: 189.99
Category
Electronics        12
Office Supplies     7
Home                6
Unknown             1
Name: count, dtype: int64
        Quantity   UnitPrice     Revenue
count  26.000000   26.000000   26.000000
mean    3.653846   35.309615   60.636538
std     2.883641   49.193441   46.836705
min     1.000000    2.250000   18.000000
25%     2.000000    8.990000   32.615000
50%     2.500000   16.875000   42.470000
75%     4.750000   39.990000   71.137500
max    12.000000  189.990000  189.990000
Orders over $100: 4
